# TASK03F — Headless Matplotlib backend for standalone Environment A processes, SAM3D_SOURCE_ROOT_VALIDATION boundary

**Technical smoke test only. Not an anthropometric accuracy benchmark.**
See `docs/experiments/TASK03C_MINIMAL_CORE_INFERENCE.md`.

A real Colab run of Task 03B's notebook hit `DEPENDENCY_ENVIRONMENT_BLOCKED` again, now with
explicit evidence: a `chumpy` wheel build failure (wrong package name — upstream's own
`INSTALL.md` specifies `chump`, verified against a fresh clone for this task), a Detectron2
wheel build failure at the pinned commit, and the environment drifting to torch `2.11.0+cu130`
/ torchvision `0.26.0+cu130` — nowhere near anything this project asked for. The strategy of
*dynamically reproducing whatever Colab's ambient kernel happens to have* is abandoned here.

This version:
- Fixes `chump` vs `chumpy` (see `sam3d_env_spec.py`'s docstring for the full verification).
- **Removes Detectron2 entirely** from this smoke test. `SAM3DBodyEstimator` supports
  `human_detector=None` natively and falls back to a full-image bounding box automatically;
  this notebook passes that box explicitly. No learned detector, no MoGe, no SAM2/SAM3 —
  those can come back one at a time later, once core inference itself is proven to run.
- **Pins an exact, fixed PyTorch stack** (torch `2.8.0` + torchvision `0.23.0`, CUDA 12.9
  wheels) instead of reading whatever Colab's default happens to be that week, and verifies
  the pin survives every subsequent install with an actual GPU tensor operation, not just a
  version-string comparison.
- **Never hides an install failure.** Every command that can fail is logged with its exact
  command, return code, stderr tail, and a specific failure category — Task 03B's notebook
  printed `Recorded failure categories: none` despite two observed build failures; that bug
  is fixed in `install_log.py`.

Structure: **PHASE A0-A5** (deterministic environment → minimal deps → checkpoint → image →
explicit-bbox inference → serialize), each independently PASS/FAIL, then the existing isolated
**PHASE B** (MHR/clad-body) only if all of Phase A passes.

Requirements: GPU runtime, and a Colab Secret named exactly `HF_TOKEN` with approved access to
`facebook/sam-3d-body-dinov3` and `facebook/sam-3d-body-vith`. Run top-to-bottom.

---

**Task 03D update:** a real Colab run with a fully working Environment A (torch/CUDA/GPU/pin all confirmed) still failed with `ModuleNotFoundError: No module named 'sam_3d_body'` -- the upstream repo root was never on the worker's module search path. Fixed with an explicit, validated `PYTHONPATH` plus a dedicated `SAM3D_SOURCE_IMPORT` pre-flight boundary (`sam3d_source_path.py`), checked before any checkpoint/model loading is attempted. See `docs/experiments/TASK03D_SAM3D_IMPORT_PATH_FIX.md`. Nothing else changed -- GPU, HF auth, checkpoint download, the torch pin, bbox logic, and Environment B are all untouched, per this task's explicit scope.

## 1. Environment inspection

In [ ]:
import platform, shutil, subprocess, sys, time, os, json

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print('Python:', platform.python_version())
print('Platform:', platform.platform())

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1e9
except ImportError:
    ram_gb = None
print('System RAM (GB):', round(ram_gb, 1) if ram_gb else 'unknown')

disk = shutil.disk_usage('/')
print(f'Disk: {disk.free/1e9:.1f} GB free / {disk.total/1e9:.1f} GB total')

gpu_query = sh('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader 2>/dev/null')
gpu_available = bool(gpu_query)
print('nvidia-smi GPU query:', gpu_query if gpu_available else 'NONE DETECTED')

# Informational only -- NOT used to pin Environment A (Task 03B tried that; it drifted to
# torch 2.11.0+cu130 / torchvision 0.26.0+cu130 in a real run). Environment A pins an exact,
# fixed version instead (section PHASE A0).
try:
    import torch as _ambient_torch
    print('Ambient kernel torch (informational, not used as a pin):', _ambient_torch.__version__)
except ImportError:
    print('Ambient kernel has no torch installed (fine -- Environment A brings its own)')

if not gpu_available:
    raise RuntimeError(
        'NO_GPU: no usable NVIDIA GPU detected. Go to Runtime > Change runtime type, select a '
        'GPU, then Runtime > Restart and run all. This notebook does not attempt full '
        'inference on CPU (Task 03 section 1).'
    )

## 2. Secure HF_TOKEN retrieval

Colab Secrets only. Never printed, logged, written to a file, or passed to
`huggingface_hub.login()` (persists to disk). Only ever used as a `token=` argument, and only
from the ambient kernel — checkpoint download happens here (needs the token); Environment A
(needs only the already-downloaded files) runs as an isolated subprocess with no token access.

In [ ]:
from google.colab import userdata

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception as exc:
    raise RuntimeError(
        "HF_AUTH_FAILURE: could not read the 'HF_TOKEN' Colab Secret. Add it via the key icon "
        "in the left sidebar, name it exactly HF_TOKEN, and enable notebook access. "
        f"Original error: {exc!r}"
    )
if not HF_TOKEN:
    raise RuntimeError('HF_AUTH_FAILURE: HF_TOKEN secret is empty.')
print(f'HF_TOKEN retrieved from Colab Secrets: OK ({len(HF_TOKEN)} chars, value not shown)')

## 3. Repository checkout and shared (torch-free) helper imports

In [ ]:
REPO_URL = 'https://github.com/eliyahumines-dot/mtm-body-checker.git'
REPO_BRANCH = 'claude/body-measurement-feasibility-4qgdrz'  # update once merged to main
REPO_DIR = '/content/mtm-body-checker'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, REPO_DIR])
else:
    print('Repo already present at', REPO_DIR)

EXPERIMENT_DIR = f'{REPO_DIR}/experiments/sam3d_mhr_clad_smoke'
if not os.path.isdir(EXPERIMENT_DIR):
    raise RuntimeError(
        f'{EXPERIMENT_DIR} not found. If this repo is private and the clone failed, manually '
        f'upload experiments/sam3d_mhr_clad_smoke/ to {EXPERIMENT_DIR} and re-run this cell.'
    )
sys.path.insert(0, EXPERIMENT_DIR)
print('Reusing Task 02/03/03B/03C code from:', EXPERIMENT_DIR)

# All plain Python + numpy -- none require torch, safe to import into the ambient kernel.
from decision_gate import PipelineState, FailureCategory, classify, phase_summary
from interchange import read_interchange, interchange_to_clad_params, InterchangeError
from install_log import InstallLog
from sam3d_env_spec import (
    TORCH_PIN, SAM3D_CORE_PIP_DEPENDENCIES, ExcludedDependencyError,
    validate_no_excluded_dependencies, pip_install_command, torch_install_command,
)
# Task 03E: sam3d_source_path validation now happens INSIDE the standalone
# _sam3d_source_import_check.py subprocess script and inside the worker itself --
# not called directly from notebook driver code, so no import needed here.

# Task 03F: every standalone Environment A subprocess launch below sanitizes its own
# env= with this (MPLBACKEND forced to Agg) as a first, redundant layer -- each of
# _sam3d_source_import_check.py and _sam3d_inference_worker.py ALSO forces this
# internally at import time regardless of what env= it was launched with (belt-and-
# suspenders, the same pattern Task 03D/03E already established for the source root).
from sam3d_matplotlib_guard import sanitized_subprocess_env
from run import measure_via_subprocess

state = PipelineState(gpu_available=gpu_available)
install_log = InstallLog()  # every failed command anywhere in Phase A/B setup is recorded here
WORK_DIR = '/content/work'
os.makedirs(WORK_DIR, exist_ok=True)

def run_shell_logged(cmd, category=None, timeout=1800, cwd=None):
    """subprocess.run wrapper that ALWAYS records failures into install_log --
    fixes Task 03B's 'Recorded failure categories: none' bug, where a failed
    command could flip a status to False with no record of why."""
    print(f'$ {cmd}')
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, cwd=cwd)
    ok = proc.returncode == 0
    if not ok:
        print(proc.stdout[-2000:])
        print(proc.stderr[-2000:])
    install_log.record(
        cmd, ok=ok, returncode=proc.returncode,
        stderr_tail=((proc.stdout or '') + '\n' + (proc.stderr or '')) if not ok else None,
        category=category,
    )
    return ok

def verify_torch_pin(python_exe, expected_torch, expected_torchvision):
    """Task 03C section 4: after each dependency group, verify torch/torchvision
    version, torch.version.cuda, torch.cuda.is_available(), AND that a real GPU
    tensor operation actually succeeds -- not just a version-string comparison."""
    check_script = (
        "import torch, torchvision\n"
        "print(torch.__version__.split('+')[0], torchvision.__version__.split('+')[0], "
        "torch.version.cuda, torch.cuda.is_available())\n"
        "x = torch.randn(4, 4, device='cuda')\n"
        "y = (x @ x).sum().item()\n"
        "print('GPU_TENSOR_OP_OK', y)\n"
    )
    proc = subprocess.run(
        [python_exe, '-c', check_script], capture_output=True, text=True, timeout=120,
        env=sanitized_subprocess_env(),
    )
    out = proc.stdout.strip()
    lines = out.splitlines()
    parts = lines[0].split() if lines else []
    torch_v = parts[0] if len(parts) > 0 else None
    torchvision_v = parts[1] if len(parts) > 1 else None
    cuda_available = parts[3] if len(parts) > 3 else None
    gpu_op_ok = any('GPU_TENSOR_OP_OK' in l for l in lines)
    result = {
        'returncode': proc.returncode,
        'torch_version': torch_v, 'torchvision_version': torchvision_v,
        'cuda_available': cuda_available, 'gpu_tensor_op_ok': gpu_op_ok,
        'torch_version_matches': torch_v == expected_torch,
        'torchvision_version_matches': torchvision_v == expected_torchvision,
        'stderr_tail': (proc.stderr or '')[-1500:] if proc.returncode != 0 else None,
    }
    result['ok'] = (
        proc.returncode == 0 and result['torch_version_matches']
        and result['torchvision_version_matches'] and gpu_op_ok
    )
    return result

# PHASE A — SAM 3D Body minimal core inference

## PHASE A0 — deterministic Python 3.11 + torch 2.8.0/cu129 environment

A dedicated venv (`/content/env_sam3d`), never the ambient Colab kernel. Pinned to an exact,
fixed version this project chose deliberately (not whatever Colab's default happens to be) —
the previous ambient-pin strategy drifted to torch `2.11.0+cu130` in a real run.

In [ ]:
ENV_A_DIR = '/content/env_sam3d'
env_a_python = f'{ENV_A_DIR}/bin/python3'

python311 = shutil.which('python3.11')
if python311 is None:
    run_shell_logged(
        'apt-get -qq update && apt-get -qq install -y python3.11 python3.11-venv python3.11-dev',
        category=FailureCategory.SAM3D_CORE_DEPENDENCY_FAILURE.value,
    )
    python311 = shutil.which('python3.11')
venv_builder = python311 or sys.executable
print('Building Environment A with:', venv_builder)

sam3d_core_environment_ok = run_shell_logged(
    f'{venv_builder} -m venv {ENV_A_DIR}', category=FailureCategory.SAM3D_CORE_DEPENDENCY_FAILURE.value,
)
sam3d_core_environment_ok &= run_shell_logged(
    f'{env_a_python} -m pip install -q --upgrade pip',
    category=FailureCategory.SAM3D_CORE_DEPENDENCY_FAILURE.value,
)
sam3d_core_environment_ok &= run_shell_logged(
    torch_install_command(env_a_python), category=FailureCategory.SAM3D_CORE_DEPENDENCY_FAILURE.value,
)

if sam3d_core_environment_ok:
    pin_check_a0 = verify_torch_pin(env_a_python, TORCH_PIN['torch'], TORCH_PIN['torchvision'])
    print(json.dumps(pin_check_a0, indent=2))
    if not pin_check_a0['ok']:
        sam3d_core_environment_ok = False
        cat = (FailureCategory.TORCH_VERSION_DRIFT if not pin_check_a0['torch_version_matches']
               else FailureCategory.TORCHVISION_VERSION_DRIFT if not pin_check_a0['torchvision_version_matches']
               else FailureCategory.SAM3D_CORE_DEPENDENCY_FAILURE)
        install_log.record('verify_torch_pin (after PHASE A0)', ok=False,
                            stderr_tail=json.dumps(pin_check_a0), category=cat.value)
        print(f'STOP: pinned torch/torchvision did not take effect as installed -- {cat.value}')
else:
    print('PHASE A0 venv/torch install failed -- skipping pin verification.')

print()
print('PHASE A0 — deterministic environment + torch pin:', 'PASS' if sam3d_core_environment_ok else 'FAIL')

## PHASE A1 — install minimal SAM 3D Body core dependencies

Exactly upstream's own `INSTALL.md` list (verified against a fresh clone for this task), minus
Detectron2, MoGe, and SAM2/SAM3 (all explicitly excluded, Task 03C sections 2-3/9), using
`chump` — not `chumpy` (Task 03B's actual observed wheel-build failure; see
`sam3d_env_spec.py`'s docstring). `validate_no_excluded_dependencies()` runs first and would
raise loudly if any of those ever reappeared in the list.

In [ ]:
sam3d_core_deps_ok = sam3d_core_environment_ok

if sam3d_core_environment_ok:
    try:
        validate_no_excluded_dependencies()
        print('Dependency-name guard passed: no chumpy/detectron2/MoGe/SAM2/SAM3 in the core list.')
        print('Installing:', ', '.join(SAM3D_CORE_PIP_DEPENDENCIES))
        sam3d_core_deps_ok &= run_shell_logged(
            pip_install_command(env_a_python), category=FailureCategory.SAM3D_CORE_DEPENDENCY_FAILURE.value,
        )
    except ExcludedDependencyError as exc:
        cat = FailureCategory.WRONG_DEPENDENCY_CHUMPY if 'chumpy' in str(exc) else FailureCategory.SAM3D_CORE_DEPENDENCY_FAILURE
        install_log.record('validate_no_excluded_dependencies()', ok=False, stderr_tail=str(exc), category=cat.value)
        sam3d_core_deps_ok = False
        print('STOP:', exc)

    if sam3d_core_deps_ok:
        pin_check_a1 = verify_torch_pin(env_a_python, TORCH_PIN['torch'], TORCH_PIN['torchvision'])
        print(json.dumps(pin_check_a1, indent=2))
        if not pin_check_a1['ok']:
            sam3d_core_deps_ok = False
            cat = (FailureCategory.TORCH_VERSION_DRIFT if not pin_check_a1['torch_version_matches']
                   else FailureCategory.TORCHVISION_VERSION_DRIFT if not pin_check_a1['torchvision_version_matches']
                   else FailureCategory.SAM3D_CORE_DEPENDENCY_FAILURE)
            install_log.record('verify_torch_pin (after PHASE A1)', ok=False,
                                stderr_tail=json.dumps(pin_check_a1), category=cat.value)
            print(f'STOP: a Phase A1 dependency mutated the torch/torchvision pin -- {cat.value}. '
                  f'Not silently resolved by upgrading torch, per Task 03C section 4.')
else:
    print('Skipping PHASE A1 -- PHASE A0 did not pass.')

state.dependencies_installed = sam3d_core_deps_ok
print()
print('PHASE A1 — minimal core dependencies:', 'PASS' if sam3d_core_deps_ok else 'FAIL')
print('SAM3D_CORE_ENVIRONMENT (PHASE A0+A1 combined):', 'PASS' if sam3d_core_deps_ok else 'FAIL')

## PHASE A1.5 — clone upstream repo + SAM3D_SOURCE_ROOT_VALIDATION / SAM3D_SOURCE_IMPORT / SAM3D_MODEL_CODE_IMPORT pre-flight check

**Root cause of a real Colab failure this fixes (Task 03D, recurred Task 03E, recurred again as a
*different* failure Task 03F fixes):** Environment A built successfully and the source root resolved
correctly (Task 03E's fix held), and then the pre-flight subprocess failed *inside*
`import sam_3d_body` itself:

```text
ValueError: Key backend: 'module://matplotlib_inline.backend_inline' is not a valid value for backend
```

Colab's interactive kernel sets `MPLBACKEND` to its own inline-plotting backend. A standalone
(non-notebook) subprocess inherits that same environment variable by default, and something SAM 3D
Body imports transitively (reproduced directly in this task: `torchmetrics`, via `pytorch_lightning`)
imports `matplotlib` as a side effect, which then fails to activate the inherited, notebook-only
backend name. **Not fixed by installing `matplotlib-inline`** — that would only mask this one
inherited value and leave a headless subprocess depending on an interactive-kernel-only package for
no reason. Fixed instead by forcing every standalone Environment A process's own `MPLBACKEND` to
`Agg` unconditionally, both as a subprocess `env=` override (`sanitized_subprocess_env()`) and,
redundantly, inside each script itself at the very top of the module — before anything that might
import matplotlib — via `sam3d_matplotlib_guard.force_headless_matplotlib_backend()`.

Also splits what Task 03D/03E called `SAM3D_SOURCE_IMPORT` into two boundaries: `SAM3D_SOURCE_ROOT_VALIDATION`
(the given root exists and has an importable `sam_3d_body/__init__.py`) is now distinct from
`SAM3D_SOURCE_IMPORT` (actually executing `import sam_3d_body`) — a real Colab run showed the root can
validate perfectly while the import itself still fails for an unrelated runtime reason (this exact
Matplotlib case), so "the root is valid" and "the root actually imports" must be independently
observable (Task 03F section 6).

Checked here, before checkpoint download or any model loading is attempted, with the exact
interpreter (`env_a_python`) that will run inference. The worker itself
(`_sam3d_inference_worker.py`) repeats this exact check internally as its own first two stages —
deliberately redundant, so a broken import is still caught even if something changed between this
pre-flight cell and the actual inference call.


In [ ]:
SAM3D_DIR = '/content/sam-3d-body'
if not os.path.isdir(SAM3D_DIR):
    run_shell_logged(
        f'git clone --depth 1 https://github.com/facebookresearch/sam-3d-body.git {SAM3D_DIR}',
        category=FailureCategory.SAM3D_CORE_DEPENDENCY_FAILURE.value,
    )

sam3d_source_root_validation_ok = False
sam3d_source_import_ok = False
sam3d_model_code_import_ok = False
sam3d_module_file = None
validated_sam3d_root = None
preflight_telemetry = {}

if sam3d_core_deps_ok:
    preflight_telemetry_path = f'{WORK_DIR}/source_import_preflight_telemetry.json'
    # Task 03E: no env=PYTHONPATH -- the script resolves/validates the root and does its own
    # sys.path.insert() internally, from this one explicit CLI argument only. Task 03F: env=
    # is sanitized here (MPLBACKEND forced to Agg) as a first, redundant layer -- the script
    # ALSO forces this itself internally at import time.
    proc = subprocess.run(
        [env_a_python, f'{EXPERIMENT_DIR}/_sam3d_source_import_check.py', SAM3D_DIR, preflight_telemetry_path],
        capture_output=True, text=True, timeout=120, env=sanitized_subprocess_env(),
    )
    print(proc.stdout)
    if proc.stderr.strip():
        print(proc.stderr[-1500:])

    if os.path.exists(preflight_telemetry_path):
        with open(preflight_telemetry_path) as f:
            preflight_telemetry = json.load(f)
    else:
        preflight_telemetry = {
            'source_root_validation_ok': False, 'source_import_ok': None, 'model_code_import_ok': None,
            'error': f'preflight script produced no telemetry file, returncode={proc.returncode}',
        }

    validated_sam3d_root = preflight_telemetry.get('sam3d_source_root_resolved')
    sam3d_module_file = preflight_telemetry.get('sam3d_module_file')
    sam3d_source_root_validation_ok = preflight_telemetry.get('source_root_validation_ok') is True
    sam3d_source_import_ok = preflight_telemetry.get('source_import_ok') is True
    sam3d_model_code_import_ok = preflight_telemetry.get('model_code_import_ok') is True

    # Task 03F: the script itself already picked the precise category (root-validation,
    # the specific Matplotlib-backend case, a general import runtime/dependency failure,
    # or model-code-import) -- read it directly rather than re-deriving it here.
    failure_category = preflight_telemetry.get('failure_category')
    if failure_category:
        install_log.record(
            f'{env_a_python} _sam3d_source_import_check.py {SAM3D_DIR}', ok=False,
            returncode=proc.returncode, stderr_tail=preflight_telemetry.get('error') or proc.stderr,
            category=failure_category,
        )
else:
    print('Skipping PHASE A1.5 -- PHASE A0/A1 did not pass.')

state.sam3d_source_root_validation_ok = sam3d_source_root_validation_ok if sam3d_core_deps_ok else None
state.sam3d_source_import_ok = (
    sam3d_source_import_ok if (sam3d_core_deps_ok and sam3d_source_root_validation_ok) else None
)
state.sam3d_model_code_import_ok = (
    sam3d_model_code_import_ok
    if (sam3d_core_deps_ok and sam3d_source_root_validation_ok and sam3d_source_import_ok) else None
)

def _p(v, gated_on):
    if gated_on is not True:
        return 'NOT_ATTEMPTED'
    return 'PASS' if v else 'FAIL'

print()
print('Inherited MPLBACKEND:        ', preflight_telemetry.get('inherited_mplbackend'))
print('Worker MPLBACKEND:           ', preflight_telemetry.get('worker_mplbackend'))
print('SAM3D_SOURCE_ROOT:           ', validated_sam3d_root or SAM3D_DIR)
print('SAM3D_MODULE_FILE:           ', sam3d_module_file)
print('SAM3D_CORE_ENVIRONMENT:      ', 'PASS' if sam3d_core_deps_ok else 'FAIL')
print('SAM3D_SOURCE_ROOT_VALIDATION:', _p(sam3d_source_root_validation_ok, sam3d_core_deps_ok))
print('SAM3D_SOURCE_IMPORT:         ', _p(sam3d_source_import_ok, sam3d_source_root_validation_ok))
print('SAM3D_MODEL_CODE_IMPORT:     ', _p(sam3d_model_code_import_ok, sam3d_source_import_ok))
if preflight_telemetry.get('matplotlib_backend_effective'):
    print('Effective matplotlib backend:', preflight_telemetry['matplotlib_backend_effective'])


## PHASE A2 — checkpoint authentication and download

Confirmed working in both real Colab runs so far (GPU/HF-auth/checkpoint-download have never
been the problem) — kept essentially unchanged. Actual `load_sam_3d_body()` model-load
PASS/FAIL is reported after PHASE A4/A5 below runs the real inference worker (which loads the
model and runs inference in one subprocess, to avoid loading multi-GB weights onto the GPU
twice in a single smoke-test run) — the granular status is still independently visible via
its own telemetry field, it is just observed at that point in execution, not before image
selection.

In [ ]:
from huggingface_hub import HfApi, snapshot_download

CHECKPOINT_REPOS = ['facebook/sam-3d-body-dinov3', 'facebook/sam-3d-body-vith']
PRIMARY_CHECKPOINT_REPO = 'facebook/sam-3d-body-dinov3'  # upstream README's own default example

api = HfApi()
access_status = {}
for repo_id in CHECKPOINT_REPOS:
    try:
        api.model_info(repo_id, token=HF_TOKEN)
        access_status[repo_id] = 'accessible'
    except Exception as exc:
        access_status[repo_id] = f'BLOCKED: {type(exc).__name__}'
print(access_status)

hf_auth_ok = access_status.get(PRIMARY_CHECKPOINT_REPO) == 'accessible'
state.hf_auth_ok = hf_auth_ok
if not hf_auth_ok:
    state.add_failure(FailureCategory.HF_AUTH_FAILURE)
    raise RuntimeError(f'CHECKPOINT_ACCESS_BLOCKED: access to {PRIMARY_CHECKPOINT_REPO} not confirmed.')

CKPT_DIR = f"/content/checkpoints/{PRIMARY_CHECKPOINT_REPO.split('/')[-1]}"
checkpoint_downloaded = False
checkpoint_download_time_s = None
checkpoint_size_bytes = None
try:
    t0 = time.time()
    snapshot_download(repo_id=PRIMARY_CHECKPOINT_REPO, local_dir=CKPT_DIR, token=HF_TOKEN)
    checkpoint_download_time_s = round(time.time() - t0, 1)
    checkpoint_downloaded = True
    checkpoint_size_bytes = int(sh(f'du -sb {CKPT_DIR}').split()[0])
    print(f'Downloaded {PRIMARY_CHECKPOINT_REPO} in {checkpoint_download_time_s}s, {checkpoint_size_bytes} bytes')
except Exception as exc:
    install_log.record(f'snapshot_download({PRIMARY_CHECKPOINT_REPO})', ok=False, stderr_tail=repr(exc),
                        category=FailureCategory.CHECKPOINT_ACCESS_FAILURE.value)
    print('CHECKPOINT_ACCESS_FAILURE:', repr(exc))
    state.add_failure(FailureCategory.CHECKPOINT_ACCESS_FAILURE)

state.checkpoint_downloaded = checkpoint_downloaded
if not checkpoint_downloaded:
    raise RuntimeError('CHECKPOINT_ACCESS_BLOCKED: checkpoint download failed after confirmed access.')

print()
print('PHASE A2 — checkpoint download:', 'PASS' if checkpoint_downloaded else 'FAIL')

## PHASE A3 — input image selection

Defaults to a public sample bundled in the official `sam-3d-body` repo. Set
`USE_SAMPLE_IMAGE = False` to upload your own photo instead — stays only in this Colab
runtime's `/content`, never written into the cloned git repo.

In [ ]:
USE_SAMPLE_IMAGE = True

SAMPLE_IMAGE = f'{SAM3D_DIR}/assets/qualitative_comparisons/sample1/input_bbox.png'

if USE_SAMPLE_IMAGE:
    input_image_path = SAMPLE_IMAGE
    used_sample_image = True
    if not os.path.exists(input_image_path):
        raise RuntimeError(f'Bundled sample image not found at {input_image_path}.')
else:
    from google.colab import files
    uploaded = files.upload()
    input_image_path = f'/content/{next(iter(uploaded))}'
    used_sample_image = False

print('Input image:', input_image_path, '(public sample)' if used_sample_image else '(user-uploaded, not committed)')
print()
print('PHASE A3 — image selection:', 'PASS' if os.path.exists(input_image_path) else 'FAIL')

## PHASE A4-A5 — explicit-bbox core inference and MHR serialization

One subprocess call to `_sam3d_inference_worker.py` (Environment A's venv) performs, in order:
model load, explicit full-image bounding-box construction and validation, `process_one_image`
with **no detector, no segmentor, no FOV estimator** (`human_detector=None`,
`human_segmentor=None`, `fov_estimator=None` — upstream's own documented no-detector path,
confirmed by source reading to be functionally identical to passing the bbox explicitly, which
this call does anyway for visibility), then `interchange.write_interchange()`. Its telemetry
reports each of Task 03C's four Phase A boundaries independently.

Gated on `sam3d_core_deps_ok and sam3d_source_root_validation_ok and sam3d_source_import_ok and sam3d_model_code_import_ok` (Task 03E, extended Task 03F) -- the worker is not launched at all if any import pre-flight boundary failed, since it would only reproduce the identical failure `_sam3d_source_import_check.py` already caught. Its own subprocess `env=` is also sanitized (`sanitized_subprocess_env()`, Task 03F) so it never inherits an invalid Colab Matplotlib backend either -- the worker forces this itself internally too, redundantly.


In [ ]:
sam3d_source_import_ok_worker = None
sam3d_model_code_import_ok_worker = None
worker_source_root_validation_ok = None
sam3d_model_load_ok = None
sam3d_core_inference_ok = None
mhr_params_serialized_ok = None
primary_telemetry = {}

if (sam3d_core_deps_ok and sam3d_source_root_validation_ok
        and sam3d_source_import_ok and sam3d_model_code_import_ok):
    interchange_path = f'{WORK_DIR}/primary_interchange.npz'
    telemetry_path = f'{WORK_DIR}/primary_telemetry.json'
    cmd = [env_a_python, f'{EXPERIMENT_DIR}/_sam3d_inference_worker.py',
           input_image_path, CKPT_DIR, validated_sam3d_root, interchange_path, telemetry_path]
    # Task 03E: no env=PYTHONPATH here either -- the worker validates sam3d_source_root itself
    # (its 4th positional argument) and does its own sys.path.insert(). Task 03F: env= is
    # sanitized (MPLBACKEND forced to Agg) as a first, redundant layer -- the worker ALSO
    # forces this itself internally at import time regardless of what it's launched with.
    proc = subprocess.run(
        cmd, capture_output=True, text=True, timeout=900, env=sanitized_subprocess_env(),
    )
    if os.path.exists(telemetry_path):
        with open(telemetry_path) as f:
            primary_telemetry = json.load(f)
    else:
        primary_telemetry = {'status': 'error', 'stage': 'worker_crashed_before_telemetry',
                              'error': f'worker produced no telemetry file, returncode={proc.returncode}',
                              'source_root_validation_ok': False, 'source_import_ok': None,
                              'model_code_import_ok': None,
                              'model_load_ok': None, 'inference_ok': None, 'interchange_written': None}
    primary_telemetry['returncode'] = proc.returncode
    if proc.returncode != 0:
        primary_telemetry['stderr_tail'] = (proc.stderr or '')[-2000:]

    print(json.dumps({k: v for k, v in primary_telemetry.items() if k != 'output_schema'}, indent=2))

    # Task 03E: preserve None ("not attempted") rather than coercing every field through
    # bool() -- the earlier bool(primary_telemetry.get(...)) pattern silently turned "this
    # stage never ran because an earlier one failed" into a reported FAIL for every
    # downstream stage (section 7's "cascading false failures" bug). `is True` / `is False`
    # only ever produce a real True/False when the worker actually recorded one; anything
    # else (missing key, or an explicit None) stays None, i.e. NOT_ATTEMPTED.
    worker_source_root_validation_ok = primary_telemetry.get('source_root_validation_ok')
    if worker_source_root_validation_ok is False:
        install_log.record('_sam3d_inference_worker.py (source root validation stage)', ok=False,
                            returncode=proc.returncode, stderr_tail=primary_telemetry.get('error') or primary_telemetry.get('stderr_tail'),
                            category=primary_telemetry.get('failure_category') or FailureCategory.SAM3D_SOURCE_ROOT_VALIDATION_FAILURE.value)
        state.sam3d_source_root_validation_ok = False

    worker_source_import_ok = primary_telemetry.get('source_import_ok')
    if worker_source_root_validation_ok is True and worker_source_import_ok is False:
        install_log.record('_sam3d_inference_worker.py (source import stage)', ok=False,
                            returncode=proc.returncode, stderr_tail=primary_telemetry.get('error') or primary_telemetry.get('stderr_tail'),
                            category=primary_telemetry.get('failure_category') or FailureCategory.SAM3D_SOURCE_IMPORT_FAILURE.value)
        state.sam3d_source_import_ok = False
    sam3d_source_import_ok_worker = worker_source_import_ok

    worker_model_code_import_ok = primary_telemetry.get('model_code_import_ok')
    if worker_source_import_ok is True and worker_model_code_import_ok is False:
        install_log.record('_sam3d_inference_worker.py (model code import stage)', ok=False,
                            returncode=proc.returncode, stderr_tail=primary_telemetry.get('error') or primary_telemetry.get('stderr_tail'),
                            category=FailureCategory.SAM3D_MODEL_CODE_IMPORT_FAILURE.value)
        state.sam3d_model_code_import_ok = False
    sam3d_model_code_import_ok_worker = worker_model_code_import_ok

    sam3d_model_load_ok = primary_telemetry.get('model_load_ok')
    sam3d_core_inference_ok = (
        primary_telemetry.get('inference_ok') if primary_telemetry.get('inference_ok') is not None else None
    )
    if sam3d_core_inference_ok is True and primary_telemetry.get('status') not in ('ok', 'ok_no_person_detected'):
        sam3d_core_inference_ok = False
    mhr_params_serialized_ok = primary_telemetry.get('interchange_written')

    if worker_model_code_import_ok is True and sam3d_model_load_ok is False:
        install_log.record('_sam3d_inference_worker.py (model load stage)', ok=False,
                            returncode=proc.returncode, stderr_tail=primary_telemetry.get('error') or primary_telemetry.get('stderr_tail'),
                            category=FailureCategory.SAM3D_MODEL_LOAD_FAILURE.value)
    elif sam3d_model_load_ok is True and sam3d_core_inference_ok is False:
        install_log.record('_sam3d_inference_worker.py (inference stage)', ok=False,
                            returncode=proc.returncode, stderr_tail=primary_telemetry.get('error') or primary_telemetry.get('stderr_tail'),
                            category=FailureCategory.SAM3D_CORE_INFERENCE_FAILURE.value)
    elif sam3d_core_inference_ok is True and not mhr_params_serialized_ok and primary_telemetry.get('person_detected'):
        install_log.record('_sam3d_inference_worker.py (interchange write stage)', ok=False,
                            stderr_tail=primary_telemetry.get('error'), category=FailureCategory.MHR_SCHEMA_FAILURE.value)
else:
    print('Skipping PHASE A4-A5 -- PHASE A0/A1 did not pass, or SAM3D_SOURCE_ROOT_VALIDATION/'
          'SAM3D_SOURCE_IMPORT/SAM3D_MODEL_CODE_IMPORT failed above.')

state.sam3d_model_load_ok = sam3d_model_load_ok
state.sam3d_inference_ok = sam3d_core_inference_ok
state.mhr_schema_valid = mhr_params_serialized_ok

def _p(v):
    return 'NOT_ATTEMPTED' if v is None else ('PASS' if v else 'FAIL')

print()
print('Inherited MPLBACKEND (worker):', primary_telemetry.get('inherited_mplbackend'))
print('Worker MPLBACKEND:            ', primary_telemetry.get('worker_mplbackend'))
print('SAM3D_SOURCE_ROOT_VALIDATION (worker):', _p(worker_source_root_validation_ok))
print('SAM3D_SOURCE_IMPORT (worker):', _p(sam3d_source_import_ok_worker))
print('SAM3D_MODEL_CODE_IMPORT (worker):', _p(sam3d_model_code_import_ok_worker))
print('SAM3D_MODEL_LOAD:      ', _p(sam3d_model_load_ok))
print('  bbox used:', primary_telemetry.get('bbox_used'), '| image:',
      primary_telemetry.get('image_width'), 'x', primary_telemetry.get('image_height'))
print('SAM3D_CORE_INFERENCE:  ', _p(sam3d_core_inference_ok),
      '| person_detected:', primary_telemetry.get('person_detected'))
print('MHR_PARAMS_SERIALIZED: ', _p(mhr_params_serialized_ok))


### Actual SAM 3D Body output schema (as observed, not assumed from docs)

In [ ]:
for k, v in primary_telemetry.get('output_schema', {}).items():
    print(f"  {k}: shape={v['shape']} dtype={v['dtype']}")
if not primary_telemetry.get('output_schema'):
    print('No output schema captured.')

mp = primary_telemetry.get('output_schema', {}).get('mhr_model_params')
sp = primary_telemetry.get('output_schema', {}).get('shape_params')
scp = primary_telemetry.get('output_schema', {}).get('scale_params')
print()
print('mhr_model_params shape:', mp['shape'] if mp else 'n/a', '(expected [204])')
print('shape_params shape:', sp['shape'] if sp else 'n/a', '(expected [45])')
if scp:
    print('scale_params shape (NOT written to the interchange file -- see adapter.py docstring):', scp['shape'])

phase_a_successful = (
    sam3d_core_deps_ok and sam3d_source_import_ok and sam3d_model_load_ok
    and sam3d_core_inference_ok and mhr_params_serialized_ok
)
print()
print('=== PHASE A required reporting (Task 03C section 7) ===')
print('SAM3D_CORE_ENVIRONMENT:', 'PASS' if sam3d_core_deps_ok else 'FAIL')
print('SAM3D_SOURCE_IMPORT:   ', 'PASS' if sam3d_source_import_ok else 'FAIL')
print('SAM3D_MODEL_LOAD:      ', 'PASS' if sam3d_model_load_ok else 'FAIL')
print('SAM3D_CORE_INFERENCE:  ', 'PASS' if sam3d_core_inference_ok else 'FAIL')
print('MHR_PARAMS_SERIALIZED: ', 'PASS' if mhr_params_serialized_ok else 'FAIL')
print('PHASE A OVERALL:       ', 'PASS' if phase_a_successful else 'FAIL')
if not phase_a_successful:
    print()
    print('Phase A did not fully pass -- Phase B (below) will not run, per Task 03C section 10.')

# PHASE B — MHR + clad-body measurement extraction

**Only runs if Phase A's four boundaries above all passed** (Task 03C section 10 — Environment
B is not investigated or modified until Phase A produces real serialized MHR parameters).
Unchanged in design from Task 03B: CPU-only, built and validated independently of Environment
A, Pixi-preferred with an automatic pip-based fallback, four-step self-test against a bundled
fixture before ever touching real Phase A output.

In [ ]:
MHR_REPO_DIR = '/content/MHR'
env_b_python = None
mhr_clad_env_build_ok = False
mhr_clad_environment_ok = False
extraction = None
raw_measurements_cm = raw_body_height_cm = calibrated_measurements_cm = calibrated_scale_factor = None
KNOWN_HEIGHT_CM = None  # e.g. 178.0 -- customer-reported height in cm, or None to skip calibration
selftest = {'ok': False, 'stage': 'not_attempted'}

if not phase_a_successful:
    print('PHASE B skipped entirely -- Phase A did not pass. This is expected behavior, not a bug',
          '(Task 03C section 10: do not investigate Environment B until Phase A succeeds).')
else:
    # --- Primary path: Pixi (MHR's own documented recommended installation method) ---
    pixi_ok = run_shell_logged('curl -fsSL https://pixi.sh/install.sh | bash',
                                category=FailureCategory.INSTALL_FAILURE.value)
    pixi_bin = os.path.expanduser('~/.pixi/bin/pixi')
    pixi_ok = pixi_ok and os.path.exists(pixi_bin)
    if pixi_ok and not os.path.isdir(MHR_REPO_DIR):
        pixi_ok &= run_shell_logged(
            f'git clone --depth 1 https://github.com/facebookresearch/MHR.git {MHR_REPO_DIR}',
            category=FailureCategory.INSTALL_FAILURE.value,
        )
    if pixi_ok:
        pixi_ok &= run_shell_logged(f'{pixi_bin} install --manifest-path {MHR_REPO_DIR}/pixi.toml',
                                      category=FailureCategory.PYMOMENTUM_FAILURE.value, timeout=1200)
    if pixi_ok:
        pixi_ok &= run_shell_logged(
            f'{pixi_bin} run --manifest-path {MHR_REPO_DIR}/pixi.toml download-assets',
            category=FailureCategory.PYMOMENTUM_FAILURE.value, timeout=900,
        )
    candidate_env_b_python = f'{MHR_REPO_DIR}/.pixi/envs/default/bin/python'
    if pixi_ok and os.path.exists(candidate_env_b_python):
        pixi_ok &= run_shell_logged(f'{candidate_env_b_python} -m pip install -q --no-deps clad-body',
                                      category=FailureCategory.CLAD_BODY_FAILURE.value)
    if pixi_ok and os.path.exists(candidate_env_b_python):
        env_b_python = candidate_env_b_python
        mhr_clad_env_build_ok = True
        print('Environment B built via Pixi at', env_b_python)
    else:
        print('Pixi path did not complete -- falling back to a pip-based CPU-torch venv.')

In [ ]:
# --- Fallback path: pip + CPU-only torch venv (only if Pixi path above did not succeed) ---
if phase_a_successful and not mhr_clad_env_build_ok:
    ENV_B_FALLBACK_DIR = '/content/env_mhr_clad_fallback'
    fallback_python = f'{ENV_B_FALLBACK_DIR}/bin/python3'
    ok = run_shell_logged(f'python3 -m venv {ENV_B_FALLBACK_DIR}', category=FailureCategory.PYMOMENTUM_FAILURE.value)
    ok &= run_shell_logged(f'{fallback_python} -m pip install -q --upgrade pip',
                            category=FailureCategory.PYMOMENTUM_FAILURE.value)
    ok &= run_shell_logged(
        f'{fallback_python} -m pip install -q torch --index-url https://download.pytorch.org/whl/cpu',
        category=FailureCategory.PYMOMENTUM_FAILURE.value,
    )
    ok &= run_shell_logged(f"{fallback_python} -m pip install -q 'clad-body[mhr]'",
                            category=FailureCategory.CLAD_BODY_FAILURE.value)
    if ok:
        import glob
        site_pkgs = glob.glob(f'{ENV_B_FALLBACK_DIR}/lib/python*/site-packages')
        assets_dir = f'{site_pkgs[0]}/assets' if site_pkgs else None
        if assets_dir and not os.path.isdir(assets_dir):
            run_shell_logged(f'mkdir -p {assets_dir}', category=FailureCategory.PYMOMENTUM_FAILURE.value)
            ok &= run_shell_logged(
                'curl -sSL -o /content/mhr_assets.zip '
                'https://github.com/facebookresearch/MHR/releases/latest/download/assets.zip',
                category=FailureCategory.PYMOMENTUM_FAILURE.value,
            )
            ok &= run_shell_logged(f'unzip -q -o /content/mhr_assets.zip -d {assets_dir}',
                                    category=FailureCategory.PYMOMENTUM_FAILURE.value)
            nested = f'{assets_dir}/assets'
            if os.path.isdir(nested):
                run_shell_logged(f'mv {nested}/* {assets_dir}/ && rmdir {nested}',
                                  category=FailureCategory.PYMOMENTUM_FAILURE.value)
    if ok:
        env_b_python = fallback_python
        mhr_clad_env_build_ok = True
        print('Environment B built via pip fallback at', env_b_python)
    else:
        print('Both the Pixi path and the pip fallback failed to build Environment B.')

if phase_a_successful:
    print()
    print('PHASE B environment build:', 'PASS' if mhr_clad_env_build_ok else 'FAIL')

## Environment B self-test (steps 1-4): import pymomentum, load MHR assets, reconstruct + measure a bundled fixture -- before touching any real SAM3D output

In [ ]:
def run_phase_b_selftest(python_exe):
    check_script = (
        "import pymomentum.geometry\n"
        "from mhr.mhr import MHR\n"
        "MHR.from_files(device='cpu', wants_pose_correctives=False)\n"
        "print('OK')\n"
    )
    proc = subprocess.run([python_exe, '-c', check_script], capture_output=True, text=True, timeout=300)
    if proc.returncode != 0:
        return {'ok': False, 'stage': 'import_pymomentum_or_load_mhr_assets',
                'returncode': proc.returncode, 'stderr_tail': (proc.stderr or '')[-1500:]}
    probe = subprocess.run(
        [python_exe, '-c', 'import clad_body, os; print(os.path.dirname(clad_body.__file__))'],
        capture_output=True, text=True, timeout=60,
    )
    if probe.returncode != 0:
        return {'ok': False, 'stage': 'locate_clad_body_package',
                'returncode': probe.returncode, 'stderr_tail': (probe.stderr or '')[-1500:]}
    fixture_path = f'{probe.stdout.strip()}/measure/testdata/mhr/female_average/mhr_params.json'
    if not os.path.exists(fixture_path):
        return {'ok': False, 'stage': 'bundled_fixture_not_found', 'fixture_path': fixture_path}
    w, fl = [], []
    status, result = measure_via_subprocess(
        fixture_path, known_height_cm=None, warnings=w, failures=fl, python_executable=python_exe,
    )
    return {'ok': status == 'ok', 'stage': 'reconstruct_and_measure_bundled_fixture',
            'status': status, 'warnings': w, 'failures': fl, 'result': result}

if phase_a_successful and mhr_clad_env_build_ok:
    selftest = run_phase_b_selftest(env_b_python)
    print(json.dumps({k: v for k, v in selftest.items() if k != 'result'}, indent=2))
    mhr_clad_environment_ok = selftest['ok']
    if not mhr_clad_environment_ok:
        stage = selftest.get('stage', '')
        cat = FailureCategory.PYMOMENTUM_FAILURE if 'pymomentum' in stage or 'mhr_assets' in stage else FailureCategory.CLAD_BODY_FAILURE
        install_log.record(f'run_phase_b_selftest ({stage})', ok=False,
                            stderr_tail=selftest.get('stderr_tail'), category=cat.value)
elif phase_a_successful:
    print('Skipping self-test -- Environment B did not build.')

state.mhr_clad_environment_ok = mhr_clad_environment_ok
if phase_a_successful:
    print()
    print('PHASE B self-test (steps 1-4, bundled fixture):', 'PASS' if mhr_clad_environment_ok else 'FAIL')

## Step 5: real SAM3D interchange data -> MHR reconstruction -> clad-body measurements

Only reached if Phase A fully passed AND Phase B's self-test passed.

In [ ]:
import tempfile

if phase_a_successful and mhr_clad_environment_ok and primary_telemetry.get('interchange_written'):
    try:
        record = read_interchange(interchange_path)
        clad_params = interchange_to_clad_params(record)
        with tempfile.NamedTemporaryFile(mode='w', suffix='_sam3d_mhr_params.json', delete=False) as f:
            json.dump(clad_params, f)
            params_path = f.name
        w, fl = [], []
        status, raw_res = measure_via_subprocess(
            params_path, known_height_cm=None, warnings=w, failures=fl, python_executable=env_b_python,
        )
        calibrated_res = None
        if status == 'ok' and KNOWN_HEIGHT_CM is not None:
            w2, fl2 = [], []
            _, calibrated_res = measure_via_subprocess(
                params_path, known_height_cm=KNOWN_HEIGHT_CM, warnings=w2, failures=fl2, python_executable=env_b_python,
            )
            w.extend(w2); fl.extend(fl2)
        os.unlink(params_path)
        extraction = {'status': status, 'raw_result': raw_res, 'calibrated_result': calibrated_res,
                      'warnings': w, 'failures': fl}
    except InterchangeError as exc:
        extraction = {'status': f'interchange_error: {exc}', 'raw_result': None, 'calibrated_result': None,
                      'warnings': [], 'failures': [str(exc)]}

    print('measurement_extraction_status:', extraction['status'])
    for w in extraction['warnings']:
        print('WARNING:', w)
    for f_ in extraction['failures']:
        print('FAILURE:', f_)
        install_log.record('measure_via_subprocess (step 5, real SAM3D data)', ok=False,
                            stderr_tail=f_, category=FailureCategory.CLAD_BODY_FAILURE.value)
    state.mhr_reconstruction_ok = extraction['status'] != 'blocked_native_crash'
    state.clad_body_measure_ok = extraction['status'] == 'ok'
    if extraction['status'] == 'ok':
        state.measurements = extraction['raw_result']['measurements_cm']
elif phase_a_successful:
    print('Skipping step 5 -- Phase A and/or Phase B self-test did not pass.')

raw_measurements_cm = extraction['raw_result']['measurements_cm'] if extraction and extraction['raw_result'] else None
raw_body_height_cm = extraction['raw_result']['raw_body_height_cm'] if extraction and extraction['raw_result'] else None
calibrated_measurements_cm = extraction['calibrated_result']['measurements_cm'] if extraction and extraction['calibrated_result'] else None
calibrated_scale_factor = extraction['calibrated_result']['rescale_factor'] if extraction and extraction['calibrated_result'] else None

if extraction:
    print()
    print('RAW body height (cm):', raw_body_height_cm)
    print('RAW measurements (cm):', json.dumps(raw_measurements_cm, indent=2) if raw_measurements_cm else None)
    if calibrated_measurements_cm:
        print(f'scale_factor = known_height / predicted_height = {KNOWN_HEIGHT_CM} / {raw_body_height_cm:.2f} = {calibrated_scale_factor:.4f}')
        print('CALIBRATED measurements (cm):', json.dumps(calibrated_measurements_cm, indent=2))
    print()
    print('PHASE B extraction (step 5, real SAM3D data):', 'PASS' if extraction['status'] == 'ok' else 'FAIL')

## Known-height calibration — note

Two independent worker calls (one `known_height_cm=None` for the raw result, one with it set
for the calibrated result — never computed by mutating the raw result). The scale factor is a
single uniform multiplier applied to every mesh vertex; corrects overall scale only, never body
proportions. See `rescale.py`'s docstring for what it does not correct.

### MTM measurement terminology mapping (reused from Task 02, unchanged)

In [ ]:
from mtm_mapping import MTM_MEASUREMENT_MAP

if raw_measurements_cm:
    for m in MTM_MEASUREMENT_MAP:
        val = raw_measurements_cm.get(m.clad_body_key) if m.clad_body_key else None
        print(f'{m.mtm_name:32s} <- {str(m.clad_body_key):20s} = {val}  [{m.confidence}]')
else:
    print('No raw measurements available to map.')

## Save results

In [ ]:
import datetime, pathlib

RESULTS_DIR_RUNTIME = '/content/results'
os.makedirs(RESULTS_DIR_RUNTIME, exist_ok=True)

output_record = {
    'subject_id': pathlib.Path(input_image_path).stem if 'input_image_path' in dir() else None,
    'used_public_sample_image': used_sample_image if 'used_sample_image' in dir() else None,
    'checkpoint_used': PRIMARY_CHECKPOINT_REPO if checkpoint_downloaded else None,
    'checkpoint_size_bytes': checkpoint_size_bytes,
    'phase_a_telemetry': {k: v for k, v in primary_telemetry.items() if k != 'output_schema'},
    'sam3d_output_schema': primary_telemetry.get('output_schema'),
    'phase_b_environment_selftest': {k: v for k, v in selftest.items() if k != 'result'},
    'raw_body_height_cm': raw_body_height_cm,
    'raw_measurements_cm': raw_measurements_cm,
    'known_height_calibration_applied': calibrated_measurements_cm is not None,
    'known_height_cm': KNOWN_HEIGHT_CM,
    'calibrated_scale_factor': calibrated_scale_factor,
    'calibrated_measurements_cm': calibrated_measurements_cm,
    'install_failures': install_log.summary(),
    'generated_at_utc': datetime.datetime.utcnow().isoformat() + 'Z',
    'note': (
        'Technical smoke test only -- NOT an anthropometric accuracy claim.'
    ),
}

runtime_output_path = f'{RESULTS_DIR_RUNTIME}/measurement_output.json'
with open(runtime_output_path, 'w') as f:
    json.dump(output_record, f, indent=2)
print('Saved (runtime-only, not committed to git):', runtime_output_path)

if used_sample_image:
    repo_results_dir = f'{EXPERIMENT_DIR}/results'
    os.makedirs(repo_results_dir, exist_ok=True)
    with open(f'{repo_results_dir}/measurement_output.json', 'w') as f:
        json.dump(output_record, f, indent=2)
    print('Also written into the cloned repo (public sample image only):',
          f'{repo_results_dir}/measurement_output.json', '-- not committed/pushed automatically.')
else:
    print('Personal image used -- result kept in the Colab runtime only.')

## Compute/runtime summary (measured, not estimated)

In [ ]:
compute_summary = {
    'environment_a': {
        'python_version': primary_telemetry.get('python_version'),
        'torch_version': primary_telemetry.get('torch_version'),
        'torch_cuda_version': primary_telemetry.get('torch_cuda_version'),
        'gpu_name': primary_telemetry.get('gpu_name'),
        'peak_vram_mb': primary_telemetry.get('peak_vram_mb'),
        'sam3d_load_time_s': primary_telemetry.get('sam3d_load_time_s'),
        'sam3d_inference_time_s': primary_telemetry.get('sam3d_inference_time_s'),
        'checkpoint_size_bytes': checkpoint_size_bytes,
        'checkpoint_download_time_s': checkpoint_download_time_s,
    },
    'environment_b': {
        'build_path': 'pixi' if env_b_python and '.pixi' in env_b_python else 'pip_fallback' if env_b_python else None,
    },
    'system_ram_gb': round(ram_gb, 1) if ram_gb else None,
    'total_install_failures_logged': len(install_log.failures),
}
print(json.dumps(compute_summary, indent=2))

## Decision gate

In [ ]:
# Fixes Task 03B's 'Recorded failure categories: none' bug: merge EVERY logged install
# failure's category into state, not just the ones a specific check happened to call
# state.add_failure() for directly.
for cat in install_log.categories():
    state.add_failure(FailureCategory(cat))

phases = phase_summary(state)
gate, reason = classify(state)

print('=== Per-phase result (Task 03C section 7/11) ===')
for name in ['SAM3D_CORE_ENVIRONMENT', 'SAM3D_SOURCE_ROOT_VALIDATION', 'SAM3D_SOURCE_IMPORT',
             'SAM3D_MODEL_CODE_IMPORT', 'SAM3D_MODEL_LOAD', 'SAM3D_CORE_INFERENCE',
             'MHR_PARAMS_SERIALIZED', 'PHASE_A_SUCCESSFUL',
             'MHR_CLAD_ENVIRONMENT', 'MHR_CLAD_EXTRACTION', 'END_TO_END']:
    print(f'{name:24s}', phases[name])
print('First failing boundary:', phases['first_failing_boundary'] or 'none -- all phases passed')
print()
print('=== Overall decision gate (Task 03 section 18) ===')
print('DECISION GATE:', gate.value, '-', gate.name)
print('Reason:', reason)
print()
print('Recorded failure categories:', state.failure_categories or 'none')
print()
print('=== Every logged install/command failure, in full (never hidden) ===')
if install_log.failures:
    print(json.dumps(install_log.summary(), indent=2))
else:
    print('none')